# Cas12a2 mRNA dosing model — keloid fibroblasts (in vitro)

A four-compartment ODE model of lipoplex delivery → mRNA → protein, simulated under
dosing every **24 / 48 / 72 h**.

> **Scope.** Parameters are eGFP-in-cell-culture values from Ligon et al. 2014 and
> Leonhardt et al. 2013. They describe the *shape* of the problem, not Cas12a2 itself.
> Replace `beta`, `T_d` and `k_TL` with your own measurements before drawing conclusions.
> Note also that Cas12a2 kills by a per-cell threshold, so protein *level* matters only
> up to the point of activation — see the final section.

## The model

$$\frac{dL_{ex}}{dt} = -\left(k_A + k_W(t)\right)L_{ex}$$
$$\frac{dL_{in}}{dt} = k_A L_{ex} - \left(k_{esc} + d_L\right)L_{in}$$
$$\frac{dM}{dt} = N\,k_{esc}L_{in} - \delta M$$
$$\frac{dP}{dt} = k_{TL}M - \left(\beta + \mu\right)P$$

| Symbol | Meaning |
|---|---|
| $L_{ex}$ | extracellular lipoplexes (bolus added at each dose) |
| $L_{in}$ | internalised lipoplexes in endosomes |
| $M$ | cytosolic mRNA |
| $P$ | protein |
| $k_W(t)$ | washout: 0 during the incubation window, large after the wash |
| $\mu=\ln 2/T_d$ | dilution by cell division — **not optional in a proliferating culture** |

The $\mu$ term is the one most often left out. Fibroblasts double on a timescale
comparable to the protein half-life, so division is a co-dominant clearance route.

In [ ]:
!pip install -q scipy matplotlib
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

## Parameters — edit these

In [ ]:
p = dict(
    k_A   = 0.27,   # lipoplex attachment / uptake, 1/h     [Ligon 2014, Table 1]
    k_esc = 0.10,   # endosomal escape (lysis), 1/h         [Ligon 2014, Table 1]
    d_L   = 0.65,   # endosomal degradation, 1/h            [Ligon 2014, Table 1]
    delta = 0.062,  # mRNA decay, 1/h  (t1/2 = 11.2 h)      [Leonhardt 2013]
    beta  = 0.056,  # protein decay, 1/h (t1/2 = 12.4 h)    [Ligon 2014, Table 1]
    k_TL  = 170.0,  # translation, 1/h                      [Ligon 2014, Table 1]
    N     = 350.0,  # mRNA molecules per lipoplex           [Ligon 2014]
    T_d   = 36.0,   # fibroblast doubling time, h           <-- MEASURE THIS
    t_w   = 4.0,    # incubation window before wash, h      <-- YOUR PROTOCOL
    k_W   = 1e3,    # washout rate once washing, 1/h        [step form: Ligon Eq. 1]
)
p["mu"] = np.log(2) / p["T_d"]
print(f"effective protein t1/2 with division: {np.log(2)/(p['beta']+p['mu']):.1f} h")

## Integration

In [ ]:
def rhs(t, y, dose_times, pr):
    Lex, Lin, M, P = y
    washing = any(td <= t < td + pr["t_w"] for td in dose_times)
    kW = 0.0 if washing else pr["k_W"]
    return [-(pr["k_A"] + kW)*Lex,
            pr["k_A"]*Lex - (pr["k_esc"] + pr["d_L"])*Lin,
            pr["N"]*pr["k_esc"]*Lin - pr["delta"]*M,
            pr["k_TL"]*M - (pr["beta"] + pr["mu"])*P]

def simulate(tau, D=1.0, t_end=240.0, pr=p):
    """Bolus D into Lex every tau hours; returns (t, P)."""
    dose_times = np.arange(0, t_end, tau)
    y, ts, Ps = [0.0]*4, [], []
    for i, td in enumerate(dose_times):
        y[0] += D
        t_stop = dose_times[i+1] if i+1 < len(dose_times) else t_end
        sol = solve_ivp(rhs, (td, t_stop), y, args=(dose_times, pr),
                        max_step=0.25, rtol=1e-8, atol=1e-10, dense_output=True)
        tt = np.linspace(td, t_stop, int((t_stop-td)*8)+1)
        ts.append(tt); Ps.append(sol.sol(tt)[3])
        y = sol.y[:, -1].tolist()
    return np.concatenate(ts), np.concatenate(Ps)

# normalise everything to the peak of a single dose
_t, _P = simulate(1e6, t_end=240.0)
norm = _P.max()
print(f"single-dose peak at t = {_t[np.argmax(_P)]:.1f} h")

## Figure

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.2))
colors = {24: "#2563eb", 48: "#d97706", 72: "#059669"}
rows = []
for tau in (24, 48, 72):
    t, P = simulate(tau, t_end=240.0); y = P/norm
    ax.plot(t, y, lw=2, color=colors[tau], label=f"every {tau} h")
    for td in np.arange(0, 240, tau):
        ax.plot(td, -0.06, marker="^", ms=5, color=colors[tau], clip_on=False)
    ss = t >= 240-tau
    rows.append((tau, y[ss].max(), y[ss].min(), y[ss].max()/y[ss].min(), y[ss].mean()))

ax.axhline(1.0, ls=":", c="0.6", lw=1)
ax.set_xlabel("Time (h)"); ax.set_ylabel("Protein  (× single-dose peak)")
ax.set_title("Expression under three dosing intervals")
ax.set_xlim(0, 240); ax.set_ylim(-0.08, None)
ax.legend(frameon=False); ax.spines[["top","right"]].set_visible(False)
ax.grid(alpha=0.25, lw=0.6); plt.show()

print(f"{'tau':>5} {'peak':>7} {'trough':>8} {'ratio':>7} {'mean':>7}")
for r in rows:
    print(f"{r[0]:5d} {r[1]:7.2f} {r[2]:8.2f} {r[3]:7.2f} {r[4]:7.2f}")

## Sensitivity: the answer hinges on $\beta$

Peak-to-trough ratio is driven almost entirely by effective protein clearance.
Sweep it before trusting any interval.

In [ ]:
for Td in (None, 48, 36, 24):
    pr = dict(p); pr["mu"] = 0.0 if Td is None else np.log(2)/Td
    lab = "no division" if Td is None else f"doubling {Td} h"
    out = []
    for tau in (24, 48, 72):
        t, P = simulate(tau, t_end=240.0, pr=pr); y = P/norm
        ss = t >= 240-tau
        out.append(f"q{tau}h {y[ss].max()/y[ss].min():5.2f}")
    print(f"{lab:>14} | eff t1/2 {np.log(2)/(pr['beta']+pr['mu']):5.1f} h | " + "  ".join(out))

## Coverage — the part that actually governs a kill switch

Cas12a2 is a per-cell threshold, not a graded response: once a cell has one activated
complex and the target transcript, it is committed. Extra doses do not deepen the
effect, they recruit **more cells**. Transfection is Poisson:

$$\text{coverage} = 1 - e^{-n\,c\,D}$$

Because $n$ and $D$ enter only as a product, **two doses at $D$ = one dose at $2D$**
for coverage. The only reason to split a dose is to stay under the carrier's
toxicity ceiling.

In [ ]:
def coverage(p_per_dose, n):
    return 1 - (1 - p_per_dose)**n

print(f"{'p/dose':>7} " + " ".join(f"{n:>7}" for n in (1,2,3,4)))
for pd in (0.3, 0.5, 0.7):
    print(f"{pd:7.1f} " + " ".join(f"{coverage(pd,n)*100:6.0f}%" for n in (1,2,3,4)))

# fit c from a GFP titration: doses (ng/well) and measured % GFP+
doses   = np.array([125, 250, 500, 1000])
pct_pos = np.array([0.12, 0.22, 0.38, 0.60])          # <-- YOUR DATA
lam = -np.log(1 - pct_pos)
c = np.linalg.lstsq(doses[:, None], lam, rcond=None)[0][0]
print(f"\nfitted c = {c:.2e} per ng/well")
print("linearity check (lambda should be proportional to dose):",
      np.round(lam/doses/c, 2))

---
### Sources
- Ligon TS, Leonhardt C, Rädler JO. *PLoS ONE* 2014;9(9):e107148. doi:10.1371/journal.pone.0107148
- Leonhardt C et al. *Nanomedicine* 2014;10(4):679–688. doi:10.1016/j.nano.2013.11.008
- Wittrup A et al. *Nat Biotechnol* 2015;33(8):870–876. doi:10.1038/nbt.3298

---
# Part 2 — Calibrated to the stated timings, plus apoptosis

**Correction carried over from Part 1.** The literature peak (20 h) and protein
half-life (12.4 h) were measured *in proliferating cultures*, so dilution by division
is already inside them. Adding a separate $\mu$ term double-counts it — that is why
Part 1's peak landed at 17.9 h instead of 20 h. Here $\mu$ is dropped and $k_{esc}$ is
solved for so the single-dose peak falls exactly at 20 h.

### Added compartments

$$\frac{dS}{dt} = -k_{act}\hat{P}(t)\,S \qquad
\frac{dC}{dt} = +k_{act}\hat{P}(t)\,S - k_{apo}C \qquad
\frac{dD}{dt} = k_{apo}C$$

| | |
|---|---|
| $S$ | alive, not yet activated |
| $C$ | committed — Cas12a2 fired, chromatin shredded, DDR running, not yet dead |
| $D$ | dead |
| $\hat P$ | protein normalised to the single-dose peak |
| $k_{apo}=1/\tau_{apo}$ | $\tau_{apo}\approx30$ h, the 24–48 h DDR→apoptosis window |

$k_{act}$ is the **unmeasured** parameter — the per-cell hazard of Cas12a2 firing per
unit protein. It is not in the literature; you fit it from your own 24/48/72 h kill data.
Everything below is a sweep over it, not a prediction.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
import matplotlib.pyplot as plt

delta, beta, k_A, d_L, t_w, k_W = 0.062, 0.056, 0.27, 0.65, 4.0, 1e3
k_apo = 1/30.0          # DDR -> apoptosis, mean 30 h  (24-48 h range)

def peak_time(k_esc):
    def rhs(t, y):
        Lex, Lin, M, P = y
        kW = 0.0 if t < t_w else k_W
        return [-(k_A+kW)*Lex, k_A*Lex-(k_esc+d_L)*Lin,
                350*k_esc*Lin-delta*M, 170*M-beta*P]
    s = solve_ivp(rhs,(0,120),[1,0,0,0],max_step=0.1,dense_output=True,
                  rtol=1e-9,atol=1e-11)
    tt = np.linspace(0,120,24001)
    return tt[np.argmax(s.sol(tt)[3])]

k_esc = brentq(lambda k: peak_time(k)-20.0, 0.02, 1.0, xtol=1e-6)
print(f"calibrated k_esc = {k_esc:.4f} /h  ->  single-dose peak at {peak_time(k_esc):.1f} h")

In [ ]:
def run(tau, k_act, Pnorm=1.0, t_end=240.0):
    doses = np.arange(0, t_end, tau); dset = list(doses)
    def rhs(t, y):
        Lex, Lin, M, P, S, C, D = y
        kW = 0.0 if any(td <= t < td+t_w for td in dset) else k_W
        commit = k_act*(P/Pnorm)*S
        return [-(k_A+kW)*Lex, k_A*Lex-(k_esc+d_L)*Lin,
                350*k_esc*Lin-delta*M, 170*M-beta*P,
                -commit, commit-k_apo*C, k_apo*C]
    y=[0,0,0,0,1.0,0,0]; ts=[]; ys=[]
    for i,td in enumerate(doses):
        y[0]+=1.0
        t2 = doses[i+1] if i+1<len(doses) else t_end
        s=solve_ivp(rhs,(td,t2),y,max_step=0.5,rtol=1e-7,atol=1e-9,dense_output=True)
        tt=np.linspace(td,t2,int((t2-td)*4)+1); ts.append(tt); ys.append(s.sol(tt))
        y=s.y[:,-1].tolist()
    return np.concatenate(ts), np.hstack(ys)

_, y1 = run(1e6, 0.0); Pnorm = y1[3].max()
t1 = np.linspace(0,240,961)
print(f"onset (10% of peak): {t1[np.argmax(run(1e6,0.0)[1][3]/Pnorm > 0.1)]:.1f} h")

### Sweep $k_{act}$ — this is where the schedule does or does not matter

In [ ]:
print(f"{'k_act':>6} {'1-dose kill':>12} |" +
      "".join(f"{'q%dh  72h / 240h'%t:>20}" for t in (24,48,72)))
rows={}
for k_act in (0.25,0.05,0.02,0.01,0.005):
    t0,y0 = run(1e6,k_act,Pnorm)
    line=f"{k_act:6.3f} {(1-y0[4][-1])*100:11.1f}% |"
    for tau in (24,48,72):
        t,y = run(tau,k_act,Pnorm); rows[(k_act,tau)]=(t,y)
        line += f"{y[6][np.argmax(t>=72)]*100:9.1f}% /{y[6][-1]*100:7.1f}%"
    print(line)

In [ ]:
fig,(a1,a2)=plt.subplots(1,2,figsize=(13,4.8))
cols={24:"#2563eb",48:"#d97706",72:"#059669"}
K = 0.01            # <-- change to the regime you care about
for tau in (24,48,72):
    t,y = run(tau,0.0,Pnorm)
    a1.plot(t,y[3]/Pnorm,lw=2,color=cols[tau],label=f"q{tau}h")
    t,y = rows[(K,tau)]
    a2.plot(t,y[6]*100,lw=2,color=cols[tau],label=f"q{tau}h")
a1.axhline(1,ls=":",c="0.6",lw=1)
a1.set(xlabel="Time (h)",ylabel="Protein (× single-dose peak)",xlim=(0,240))
a1.set_title("Protein — calibrated: onset 3 h, peak 20 h")
a2.set(xlabel="Time (h)",ylabel="Cumulative dead cells (%)",xlim=(0,240),ylim=(0,100))
a2.set_title(f"Apoptosis, $k_{{act}}$={K}")
for a in (a1,a2):
    a.legend(frameon=False); a.spines[["top","right"]].set_visible(False)
    a.grid(alpha=.25,lw=.6)
plt.show()

### Fitting $k_{act}$ to your data

Measure cumulative death (annexin V⁺ or viability loss, vehicle-subtracted) at 24, 48
and 72 h after a **single** dose, put it in `obs` below, and this returns the $k_{act}$
that reproduces it. Then re-run the sweep to see what the schedules predict.

In [ ]:
from scipy.optimize import minimize_scalar

t_obs = np.array([24., 48., 72.])
obs   = np.array([0.08, 0.22, 0.35])      # <-- YOUR vehicle-subtracted fraction dead

def sse(log_k):
    t,y = run(1e6, 10**log_k, Pnorm)
    pred = np.interp(t_obs, t, y[6])
    return float(np.sum((pred-obs)**2))

r = minimize_scalar(sse, bounds=(-4,0), method="bounded")
k_fit = 10**r.x
t,y = run(1e6, k_fit, Pnorm)
print(f"fitted k_act = {k_fit:.4f}   SSE = {r.fun:.5f}")
print("predicted:", np.round(np.interp(t_obs,t,y[6]),3), " observed:", obs)